# Giá niêm yết xe đã qua sử dụng tại UAE

**Môn:** ADY201m — Data Science with Python & SQL  
**Loại bài:** project phân tích dữ liệu khám phá và baseline hồi quy  
**Nhóm:** điền tên và mã sinh viên trước khi nộp

> Đề/rubric chính thức không được cung cấp trong yêu cầu hiện tại. Notebook này triển khai project học thuật theo các chủ đề môn học đã nêu; cần đối chiếu với giảng viên trước khi coi là đúng định dạng nộp cuối cùng.

**Câu hỏi chính:** Các thuộc tính xe và địa điểm liên hệ như thế nào với giá niêm yết xe cũ trong bộ dữ liệu UAE?

**Nguồn:** [Kaggle — UAE Used Car Prices & Features (10K+ Listings)](https://www.kaggle.com/datasets/alikalwar/uae-used-car-prices-and-features-10k-listings), Ali Kalwar, license CC BY-SA 4.0. `Price` được xem là giá trong tin đăng, không khẳng định là giá giao dịch.


## 1. Xác định bài toán và research questions

- **RQ1:** Giá niêm yết phân bố thế nào và thay đổi ra sao theo năm, số km, hãng, loại thân xe, địa điểm?
- **RQ2:** Trung vị giá của SUV và Sedan có khác nhau trong mẫu không? Dùng Mann–Whitney U hai phía vì giá lệch phải và có ngoại lệ.
- **RQ3:** Mô hình tuyến tính baseline dùng `Year` và `Mileage` dự đoán giá trên hold-out ra sao so với baseline dự đoán trung vị?

Đây là dữ liệu quan sát; kiểm định và hồi quy chỉ mô tả liên hệ, không chứng minh nguyên nhân.


## 2. Nạp và làm sạch dữ liệu

Project dùng pandas/numpy cho phân tích, SQLite cho SQL. Mã nguồn phân tích nằm trong `run_analysis.py`; CSV gốc không bị thay đổi. Bước làm sạch chuẩn hóa khoảng trắng, chuyển trường số và coi ô trống, `None`, `Unknown` là thiếu ở `Cylinders`.


In [4]:
from pathlib import Path
import sys, json, sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()), Path.cwd())
ANALYSIS_DIR = PROJECT_ROOT / "scripts" / "analysis"
sys.path.insert(0, str(ANALYSIS_DIR))
from run_analysis import load_and_clean, create_database, model_evaluation, mann_whitney_approx, run

results = run()  # Nạp CSV, làm sạch, dựng SQLite, chạy thống kê và lưu kết quả tái lập
df, audit = load_and_clean()
print("Data shape:", df.shape)
print("Columns:", list(df.columns))
print("Cleaning audit:", json.dumps(audit, ensure_ascii=False, indent=2))
print(df.head(5).to_string(index=False))


Data shape: (10000, 13)
Columns: ['Make', 'Model', 'Year', 'Price', 'Mileage', 'Body Type', 'Cylinders', 'Transmission', 'Fuel Type', 'Color', 'Location', 'Description', 'Listing_ID']
Cleaning audit: {
  "rows_source": 10000,
  "rows_analysis": 10000,
  "columns": [
    "Make",
    "Model",
    "Year",
    "Price",
    "Mileage",
    "Body Type",
    "Cylinders",
    "Transmission",
    "Fuel Type",
    "Color",
    "Location",
    "Description"
  ],
  "duplicate_rows_removed": 0,
  "missing_after_normalization": {
    "Make": 0,
    "Model": 0,
    "Year": 0,
    "Price": 0,
    "Mileage": 0,
    "Body Type": 0,
    "Cylinders": 155,
    "Transmission": 0,
    "Fuel Type": 0,
    "Color": 0,
    "Location": 0,
    "Description": 0
  },
  "cylinders_missing_normalized": 155,
  "location_values_trimmed": 9869
}
  Make                  Model  Year  Price  Mileage            Body Type  Cylinders           Transmission Fuel Type Color  Location                                              

### Từ điển dữ liệu

| Cột | Ý nghĩa | Kiểu phân tích |
|---|---|---|
| `Make` | Hãng xe | Phân loại |
| `Model` | Mẫu xe | Phân loại |
| `Year` | Năm sản xuất | Số nguyên |
| `Price` | Giá đăng bán, AED | Số thực; biến kết quả |
| `Mileage` | Số km đã đi | Số nguyên |
| `Body Type` | Dạng thân xe | Phân loại |
| `Cylinders` | Số xi-lanh; có giá trị chưa biết | Số nguyên/thiếu |
| `Transmission` | Loại hộp số | Phân loại |
| `Fuel Type` | Loại nhiên liệu | Phân loại |
| `Color` | Màu xe | Phân loại |
| `Location` | Địa điểm UAE | Phân loại; trim khoảng trắng |
| `Description` | Nội dung tự do, gồm tiện nghi/tình trạng | Văn bản |

**Kết quả profile nguồn này:** 10.000 dòng, 12 cột, không có dòng trùng hoàn toàn; sau chuẩn hóa có 155 `Cylinders` thiếu/không xác định; 9.869 giá trị `Location` có khoảng trắng đầu/cuối.


## 3. SQL: truy vấn qua SQLite

Database chuẩn hóa các bảng `make`, `body_type`, `emirate`, `car_listing`, dùng khóa ngoại và chỉ mục. Các truy vấn dưới đây thể hiện `JOIN`, `GROUP BY`, `HAVING`, `ORDER BY`.


In [7]:
db_path = PROJECT_ROOT / "database" / "uae_used_cars.db"
conn = sqlite3.connect(db_path)
q_make = """SELECT m.make_name AS make, COUNT(*) AS listings, ROUND(AVG(c.price_aed), 0) AS mean_price_aed
FROM car_listing c JOIN make m ON c.make_id=m.make_id
GROUP BY m.make_name HAVING COUNT(*) >= 50 ORDER BY mean_price_aed DESC LIMIT 10"""
q_body = """SELECT b.body_type_name AS body_type, COUNT(*) AS listings, ROUND(AVG(c.price_aed), 0) AS mean_price_aed
FROM car_listing c JOIN body_type b ON c.body_type_id=b.body_type_id
GROUP BY b.body_type_name ORDER BY listings DESC"""
print("Top makes by average asking price (minimum 50 listings):")
print(pd.read_sql_query(q_make, conn).to_string(index=False))
print("\nPrice summary by body type:")
print(pd.read_sql_query(q_body, conn).to_string(index=False))
conn.close()


Top makes by average asking price (minimum 50 listings):
         make  listings  mean_price_aed
  rolls-royce       141       1827530.0
  lamborghini       112       1608101.0
      ferrari       123       1225834.0
      bentley       138        560116.0
mercedes-benz      1486        372082.0
      porsche       450        336755.0
   land-rover       538        249111.0
        tesla        53        249065.0
     maserati        87        247523.0
     cadillac       147        212006.0

Price summary by body type:
           body_type  listings  mean_price_aed
                 SUV      4607        259642.0
               Sedan      2790        156058.0
               Coupe       837        369481.0
       Pick Up Truck       432        148241.0
           Hatchback       334         78667.0
          Sports Car       270        706251.0
                 Van       143         99186.0
Hard Top Convertible       139        660750.0
           Crossover       136        116446.0
Soft

SQL `AVG` is sensitive to very expensive listings. For the main descriptive comparisons, also inspect medians and sample sizes. Do not rank a make with very few listings as if its average were stable.


In [9]:
# Descriptive statistics and medians by selected groups
print("Price (AED):")
print(df["Price"].describe(percentiles=[.25, .5, .75]).round(1).to_string())
print("\nMedian asking price by body type (with sample size):")
print(df.groupby("Body Type")["Price"].agg(["count", "median", "mean"]).sort_values("median", ascending=False).round(0).to_string())
print("\nMedian asking price by year:")
print(df.groupby("Year")["Price"].agg(["count", "median"]).round(0).to_string())
print("\nPearson correlations (numerical fields):")
print(df[["Year", "Mileage", "Price"]].corr(method="pearson").round(3).to_string())
print("\nSpearman rank correlations:")
print(df[["Year", "Mileage", "Price"]].corr(method="spearman").round(3).to_string())


Price (AED):
count       10000.0
mean       245234.5
std        470977.3
min          7183.0
25%         50352.5
50%        102766.0
75%        231248.0
max      14686975.0

Median asking price by body type (with sample size):
                      count    median      mean
Body Type                                      
Sports Car              270  248578.0  706251.0
Soft Top Convertible    120  224186.0  569322.0
Coupe                   837  214211.0  369481.0
Wagon                    44  155982.0  308645.0
Hard Top Convertible    139  140062.0  660750.0
SUV                    4607  118414.0  259642.0
Pick Up Truck           432  113958.0  148241.0
Utility Truck            34  109185.0  234184.0
Crossover               136   86282.0  116446.0
Other                   114   82957.0  164798.0
Sedan                  2790   70288.0  156058.0
Van                     143   51966.0   99186.0
Hatchback               334   50980.0   78667.0

Median asking price by year:
      count    median
Y

## 4. Kiểm định giả thuyết: SUV và Sedan

- **H₀:** phân phối giá niêm yết của SUV và Sedan giống nhau.
- **H₁:** hai phân phối khác nhau.
- Chọn mức ý nghĩa 0,05. Mann–Whitney U dùng xấp xỉ chuẩn có hiệu chỉnh ties, hai phía; p-value rất nhỏ vẫn không đo mức chênh lệch thực tế và không loại trừ yếu tố gây nhiễu như hãng/năm.


In [11]:
suv = df.loc[df["Body Type"] == "SUV", "Price"].to_numpy(dtype=float)
sedan = df.loc[df["Body Type"] == "Sedan", "Price"].to_numpy(dtype=float)
test = mann_whitney_approx(suv, sedan)
print("SUV n, median AED:", len(suv), float(np.median(suv)))
print("Sedan n, median AED:", len(sedan), float(np.median(sedan)))
print("Mann-Whitney U (normal approximation):", json.dumps(test, indent=2))
print("Decision at alpha=0.05:", "reject H0" if test["p_value_approx"] < .05 else "fail to reject H0")


SUV n, median AED: 4607 118414.0
Sedan n, median AED: 2790 70287.5
Mann-Whitney U (normal approximation): {
  "U": 4822934.5,
  "z_approx": -18.01692143915931,
  "p_value_approx": 1.4351044328544343e-72,
  "n_group1": 4607,
  "n_group2": 2790
}
Decision at alpha=0.05: reject H0


## 5. Baseline hồi quy và đánh giá

Dùng 80% train / 20% test, chia ngẫu nhiên có seed cố định. Dự đoán `log(Price)` bằng hồi quy bình phương tối thiểu với `Year` và `Mileage`; sau đó đổi dự đoán về AED. So sánh với mô hình mốc luôn dự đoán trung vị giá train. Vì phân phối giá có đuôi phải rất dài, đọc đồng thời MAE, RMSE và R² trên AED.


In [13]:
model_result = model_evaluation(df)
print(json.dumps(model_result, ensure_ascii=False, indent=2))


{
  "n_train": 8000,
  "n_test": 2000,
  "predictors": [
    "Year",
    "Mileage"
  ],
  "target_transform": "natural log of Price",
  "coefficients_log_price": {
    "intercept": 11.66091534123378,
    "Year_standardized": 0.004263688122460233,
    "Mileage_standardized": 0.008780865098024037
  },
  "baseline_train_median": {
    "MAE_AED": 193477.773,
    "RMSE_AED": 488598.70034067205,
    "R2_AED": -0.09603896306465654
  },
  "linear_regression": {
    "MAE_AED": 193895.59373891552,
    "RMSE_AED": 484787.6861075128,
    "R2_AED": -0.0790076858841049
  },
  "note": "Metrics are on AED scale after exponentiating predicted log price; a simple baseline, not a valuation product."
}


## 6. Biểu đồ và kiểm tra ngoại lệ

Khi `matplotlib` và `seaborn` có sẵn, `run_analysis.py` lưu hai PNG. Nếu môi trường hiện tại thiếu chúng, script tạo SVG thay thế để vẫn xem được mẫu quan hệ và trung vị theo năm. Giá tối đa lớn hơn rất nhiều trung vị; không xóa ngoại lệ nếu chưa kiểm tra bản ghi và lý do nghiệp vụ.

![Giá theo số km, trục giá log](../../docs/figures/price_vs_mileage.svg)

![Trung vị giá theo năm](../../docs/figures/median_price_by_year.svg)

Để có biểu đồ notebook-native với seaborn, cài `requirements.txt` rồi chạy lại notebook.


In [15]:
print("Price quantiles:")
print(df["Price"].quantile([0, .25, .5, .75, .9, .95, .99, 1]).to_string())
print("\nChart files:")
for p in sorted((PROJECT_ROOT / "docs" / "figures").glob("*.svg")) + sorted((PROJECT_ROOT / "docs" / "figures").glob("*.png")):
    print(p.relative_to(PROJECT_ROOT))


Price quantiles:
0.00        7183.00
0.25       50352.50
0.50      102766.00
0.75      231248.00
0.90      553848.40
0.95     1052589.00
0.99     2192676.66
1.00    14686975.00

Chart files:
docs\figures\median_price_by_year.svg
docs\figures\price_vs_mileage.svg
docs\figures\median_price_by_year.png
docs\figures\price_vs_mileage.png


## 7. Kết luận và giới hạn

Trong mẫu này, giá trung vị là khoảng 102.766 AED; SUV có trung vị 118.414 AED, Sedan 70.287,5 AED. Kiểm định phân phối cho p-value xấp xỉ 1,44×10⁻⁷², phù hợp với sự khác biệt trong mẫu, nhưng không chứng minh loại thân xe tự nó làm thay đổi giá.

Tương quan tuyến tính thô giữa giá và năm/km gần 0; điều này cho thấy hai biến đơn lẻ chưa giải thích tốt giá của nhiều mẫu xe khác nhau. Baseline hồi quy Year+Mileage có MAE xấp xỉ 193.896 AED và R² âm trên tập test (−0,079), nên trong bộ dữ liệu này nó chưa dự đoán tốt hơn nhiều so với biến thiên quanh trung bình. Model rất đơn giản; chưa mã hóa hãng/mẫu, thân xe, địa điểm, nhiên liệu, tình trạng, cũng chưa xử lý outlier hoặc đánh giá chéo.

**Giới hạn:** tin đăng không phải giao dịch xác nhận; nguồn/mẫu có thể thiên lệch; `Description` cần trích xuất có kiểm soát; `Cylinders` thiếu 155 giá trị; nhóm hiếm có kết quả không ổn định; khoảng giá cực rộng cần xác minh. Không đưa ra khuyến nghị định giá xe cá nhân từ baseline này.
